In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model_on_test_data():
    try:
        # 1. Fresh master dataset aur trained model load karo
        df = pd.read_csv('train.csv')
        model = joblib.load('house_model_pipeline.pkl')
        print("🚀 [1] Master dataset 'train.csv' aur saved model capsule load ho gaya.")
    except FileNotFoundError:
        print("❌ Error: 'train.csv' ya 'house_model_pipeline.pkl' missing hai. Pehle training run karein.")
        return

    # 2. FEATURE ENGINEERING (Exactly same columns create karein)
    print("🛠️ [2] Testing features matrix extract ho rahi hai...")
    df['TotalSF'] = df['GrLivArea'] + df.get('TotalBsmtSF', 0)
    df['TotalBath'] = df.get('FullBath', 0) + 0.5 * df.get('HalfBath', 0) + df.get('BsmtFullBath', 0)
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)

    # Features Blueprint
    NUM_FEATURES = ['TotalSF', 'LotArea', 'HouseAge', 'GarageCars', 'Fireplaces', 'TotalBath', 'IsRemodeled']
    CAT_FEATURES = ['Neighborhood', 'SaleCondition']
    FEATURES = NUM_FEATURES + CAT_FEATURES
    TARGET = 'SalePrice'

    X = df[FEATURES]
    y_raw = df[TARGET]

    # 3. SPLIT TO GET THE EXACT UNSEEN TEST DATA (20%)
    # random_state=42 rakhne se ye strictly wahi 20% data uthayega jo training me model se chhupaya gaya tha
    _, X_test, _, y_test_raw = train_test_split(X, y_raw, test_size=0.2, random_state=42, shuffle=True)
    print(f"📋 [3] Model ko check karne ke liye {len(X_test)} unseen test ghar alag kar liye gaye hain.")

    # 4. CHUPCHAAP INFERENCE/PREDICTION RUN KARO
    log_predictions = model.predict(X_test)
    final_prices = np.expm1(log_predictions) # Log values ko real dollar prices me convert kiya

    # 5. SCORES REPORT CARD (Real-time visible validation metrics)
    print("\n📈 --- UNSEEN TESTING SCORES REPORT CARD --- 📈")
    
    r2 = r2_score(y_test_raw, final_prices)
    mae = mean_absolute_error(y_test_raw, final_prices)
    rmse = np.sqrt(mean_squared_error(y_test_raw, final_prices))
    
    print(f"✅ Model Testing Accuracy (R² Score): {r2:.4f} (~{r2*100:.2f}%)")
    print(f"✅ Mean Absolute Error (MAE):         ${mae:,.2f}")
    print(f"✅ Root Mean Squared Error (RMSE):     ${rmse:,.2f}")
    print("---------------------------------------------------\n")

    # 6. EXCEL FILE MEIN SAVE KARO (Aapke verification ke liye)
    output_test_sheet = X_test.copy()
    output_test_sheet['Actual_SalePrice'] = y_test_raw
    output_test_sheet['Predicted_SalePrice'] = np.round(final_prices, 2)
    output_test_sheet['Absolute_Error'] = np.abs(output_test_sheet['Actual_SalePrice'] - output_test_sheet['Predicted_SalePrice'])
    
    output_test_sheet.to_csv('test_predictions_result.csv', index=False)
    print("💾 [6] Saare answers comparing sheet ke sath 'test_predictions_result.csv' me save ho gaye hain!\n")

if __name__ == "__main__":
    evaluate_model_on_test_data()

🚀 [1] Master dataset 'train.csv' aur saved model capsule load ho gaya.
🛠️ [2] Testing features matrix extract ho rahi hai...
📋 [3] Model ko check karne ke liye 292 unseen test ghar alag kar liye gaye hain.

📈 --- UNSEEN TESTING SCORES REPORT CARD --- 📈
✅ Model Testing Accuracy (R² Score): 0.2181 (~21.81%)
✅ Mean Absolute Error (MAE):         $62,382.85
✅ Root Mean Squared Error (RMSE):     $77,445.04
---------------------------------------------------

💾 [6] Saare answers comparing sheet ke sath 'test_predictions_result.csv' me save ho gaye hain!

